## Camada Silver


### Leitura e Tratamento das Tabelas:
- `(catalogo).(bronze_db_name).dm_base_atendentes`
- `(catalogo).(bronze_db_name).dm_base_motivos`
- `(catalogo).(bronze_db_name).dm_canais`
- `(catalogo).(bronze_db_name).ft_chamados`
- `(catalogo).(bronze_db_name).ft_clientes`
- `(catalogo).(bronze_db_name).ft_custos`
- `(catalogo).(bronze_db_name).ft_chamados_hora`
- `(catalogo).(bronze_db_name).ft_pesquisa_satisfacao`

#### Principais etapas de tratamento realizadas nas tabelas:

- **Padronização de nomes de colunas**: Todos os nomes foram convertidos para letras minúsculas para garantir consistência.
- **Detecção e remoção de valores nulos**: Foram identificados e excluídos registros com valores nulos em campos essenciais.
- **Formatação de dados**: Remoção de caracteres indesejados e ajuste de formatos, especialmente em colunas de data e hora.
- **Conversão de tipos de dados**: Colunas de datas e horários foram convertidas para o tipo `timestamp`
- **Criação de colunas derivadas**: Foram criadas colunas de tempo (ex: tempo de espera, tempo de atendimento) a partir dos timestamps.
- **Validação de integridade temporal**: Registros com diferenças de tempo negativas ou inconsistentes foram removidos.
- **Verificação de identificadores**: Garantia de que os campos de identificadores seguem o padrão esperado.

### Criação da Tabela `ft_chamados_geral` (*one big table*):

- `(catalogo).(silver_db_name).ft_chamados_geral`

### Tabelas retornadas para uso na camada gold:

- `(catalogo).(silver_db_name).dm_base_atendentes`
- `(catalogo).(silver_db_name).dm_base_motivos`
- `(catalogo).(silver_db_name).dm_canais`
- `(catalogo).(silver_db_name).ft_chamados`
- `(catalogo).(silver_db_name).ft_clientes`
- `(catalogo).(silver_db_name).ft_custos`
- `(catalogo).(silver_db_name).ft_chamados_hora`
- `(catalogo).(silver_db_name).ft_pesquisa_satisfacao`
- `(catalogo).(silver_db_name).ft_chamados_geral`

## 1. Setup do ambiente

### 1.1 Importação de Bibliotecas
Importação de funções essenciais do **PySpark**, 
Selecionamos especificamente a função (`current_timestamp`) para a criação da coluna `data_criacao_silver` na tabela criada `chamados_geral`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, col, trim, regexp_replace, initcap, when, lower, lit, udf
from pyspark.sql.types import StringType, IntegerType, LongType, TimestampType, DecimalType
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from functools import reduce

catalogo = 'medalhao_credit'
bronze_db_name = 'bronze_credit'
silver_db_name = 'silver_credit' 

### 1.2 Configuração de Ambiente

Definição do catálogo `catalogo` e o schema `silver_db_name` que serão utilizados. 

In [0]:
spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {silver_db_name}")

## 2. Funções Úteis

### Função `table_check(table_name, db_name)`

Esta função verifica se uma tabela existe em um banco de dados específico e se ela contém dados.

In [0]:
def table_check(table_name, db_name):
    """
    Verifica se uma tabela existe e possui dados em um banco de dados especificado.

    Args:
        table_name (str): Nome da tabela a ser verificada.
        db_name (str): Nome do banco de dados onde a tabela está localizada.

    Returns:
        bool: True se a tabela existe e possui dados, False caso contrário.

    Raises:
        ValueError: Se a tabela existe mas está vazia.
    """
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):
        if spark.table(f"{db_name}.{table_name}").count() == 0:
            raise ValueError(f"Tabela {db_name}.{table_name} existe mas está vazia.")
        return True
    return False

### Função `save_table_silver` 
Salva uma tabela a partir do catálogo (`catalogo`) e camada (`silver_db_name`) especificados, em formato Delta.
Adiciona a coluna `data_tratamento_silver`

In [0]:
def save_table_silver(table_name: str, df, process_col=True):
    """
    Salva um DataFrame como tabela Delta na camada silver.

    Args:
        table_name (str): Nome da tabela a ser criada ou sobrescrita na camada silver.
        df (DataFrame): DataFrame Spark que será salvo como tabela Delta.
        process_col(Bool): Adicionar uma coluna com o current timestamp.
    """
    table_path = f"{catalogo}.{silver_db_name}.{table_name}"

    try:
        old_schema = spark.table(table_path).schema.simpleString()
    except AnalysisException:
        old_schema = None

    if process_col == True:
        df = df.withColumn("data_tratamento_silver", current_timestamp())
    
    try:
        df.write \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(table_path)
    except Exception as ex:
        print(f"Erro ao salvar a tabela {table_path}: {ex}")
        return
    
    new_schema = spark.table(table_path).schema.simpleString()

    if old_schema is None:
        print(f"Tabela {table_path} criada pela primeira vez.")
    elif old_schema != new_schema:
        print(f"Esquema da tabela {table_path} foi alterado.\n")
        print("Schema anterior:")
        print(old_schema)
        print("\nNovo schema:")
        print(new_schema)
    else:
        print(f"Tabela salva com sucesso: {table_path}")

### Função `read_table`
Lê uma tabela do Databricks a partir do catálogo e camada especificados (`bronze` ou `silver`), retornando um DataFrame Spark correspondente.

In [0]:
def read_table(nome_tabela: str, camada:str='bronze'):
    """
    Lê uma tabela Delta da camada especificada.

    Args:
        nome_tabela (str): Nome da tabela a ser lida.
        camada (str, optional): Camada de origem da tabela ('bronze' ou 'silver'). Default é 'bronze'.

    Returns:
        DataFrame: DataFrame Spark da tabela lida.

    Raises:
        ValueError: Se a camada não for 'bronze' ou 'silver'.
        ValueError: Se a tabela não existir ou estiver vazia.
    """
    db_map = {
        'bronze': bronze_db_name,
        'silver': silver_db_name,
    }

    db_nome = db_map.get(camada.lower())
    
    if not db_nome:
        raise ValueError("Camada deve ser 'bronze' ou 'silver'")
    
    if not table_check(nome_tabela, db_nome):
        raise ValueError(f"Tabela {db_nome}.{nome_tabela} não existe.")
    return spark.table(f"{catalogo}.{db_nome}.{nome_tabela}")

### Função `describe_table`
Exibe o schema, contagem de linhas com valores nulos e 5 linhas da tabela.

In [0]:
def describe_table(df):
    """
    Exibe o schema, linhas com valores nulos e as primeiras 5 linhas do DataFrame usando display.

    Args:
        df (DataFrame): DataFrame Spark a ser exibido.
    """
    df.printSchema()
    print(f"Total de linhas: {df.count()}")
    null_count = df.filter(
        reduce(lambda a, b: a | b, [F.col(c).isNull() for c in df.columns])
    ).count()
    print(f"Linhas com valores nulos: {null_count}")
    display(df.limit(5))

## 3. Tratamento da tabela `ft_chamados_hora` 

### 3.1 Tabela `ft_chamados_hora` na camada bronze

A tabela armazena informações sobre chamados de atendimento na camada bronze do Data Lake. A tabela contém os seguintes campos:

- **ID_Chamado**: Identificador único do chamado.
- **ID_Cliente**: Identificador do cliente relacionado ao chamado.
- **Hora_Abertura_Chamado**: Data e hora em que o chamado foi aberto.
- **Hora_Inicio_Atendimento**: Data e hora de início do atendimento do chamado.
- **Hora_Finalizacao_Atendimento**: Data e hora de finalização do atendimento.
- **data_ingestao**: Data de ingestão do registro na camada bronze.

In [0]:
df_ft_chamados_hora = read_table('ft_chamados_hora', 'bronze')
describe_table(df_ft_chamados_hora)

### 3.2 Tratamento de nomes das colunas na tabela `ft_chamados_hora`

- Tratamento do header
- Os nomes das colunas do DataFrame foram convertidos para letras minúsculas, garantindo padronização.

In [0]:
nomes_colunas = [
    "id_chamado",
    "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "data_ingestao"
                ]

df_ft_chamados_hora = df_ft_chamados_hora.toDF(*nomes_colunas)

describe_table(df_ft_chamados_hora)

### 3.3 Verificação dos dados da tabela `ft_chamados_hora` 

- **Verificação de valores nulos**: Verifica linhas onde qualquer uma das colunas essenciais (`hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`, `data_ingestao`) possui valor nulo.
- **Verificação de linhas com tempos inconsistentes**: Verifica registros onde os cálculos de tempo (`tempo_espera_seg`, `tempo_atendimento_seg`, `diff_abertura_ingestao_seg`) resultam em valores negativos, indicando inconsistência temporal.
- **Verificação de valores irregulares em identificadores**: Verifica linhas onde os campos `id_cliente` e `id_chamado` são nulos ou não seguem o padrão numérico esperado.

In [0]:
print(f'linhas em ft_chamados_hora: {df_ft_chamados_hora.count()}')

In [0]:
df_ft_chamados_hora_null = df_ft_chamados_hora.filter(
    F.col('hora_abertura_chamado').isNull() |
    F.col('hora_inicio_atendimento').isNull() |
    F.col('hora_finalizacao_atendimento').isNull() |
    F.col('data_ingestao').isNull()
)

if df_ft_chamados_hora_null.count() == 0:
    print('valores nulos: 0')
else:
    df_ft_chamados_hora_null.limit(5).display()

#### 2.3.1 Tratamento de valores na tabela `ft_chamados_hora`

- As colunas `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento` passaram por duas etapas:
  1. Remoção de caracteres indesejados (" às ") usando `regexp_replace` para limpar os valores.
  2. Conversão dos valores dessas colunas para o tipo `timestamp`.

In [0]:
hora_cols = ['hora_abertura_chamado', 'hora_inicio_atendimento', 'hora_finalizacao_atendimento']

for coluna in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        coluna, F.regexp_replace(F.col(coluna), r' �s ', ' ')
    )

for coluna in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        coluna, F.to_timestamp(coluna, 'dd/MM/yyyy HH:mm:ss')
    )

#### 2.3.2 Criação de colunas na tabela `ft_chamados_hora`

- **tempo_espera_seg**: tempo entre a abertura do chamado e o início do atendimento.
- **tempo_atendimento_seg**: tempo entre o início e a finalização do atendimento.
- **diff_abertura_ingestao_seg**: tempo entre a abertura do chamado e o momento de ingestão do registro na base.

Os cálculos são feitos convertendo os timestamps para o tipo `long` (segundos desde a época Unix) e subtraindo os valores correspondentes.

In [0]:
df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
    'tempo_espera_seg',
    (F.col('hora_inicio_atendimento').cast('long') - F.col('hora_abertura_chamado').cast('long'))
).withColumn(
    'tempo_atendimento_seg',
    (F.col('hora_finalizacao_atendimento').cast('long') - F.col('hora_inicio_atendimento').cast('long'))
).withColumn(
    'diff_abertura_ingestao_seg',
    (F.col('data_ingestao').cast('long') - F.col('hora_abertura_chamado').cast('long'))
)

In [0]:
describe_table(df_ft_chamados_hora)

In [0]:
df_ft_chamados_hora_tempo_dif = df_ft_chamados_hora.filter(
    (F.col('tempo_espera_seg') < 0) |
    (F.col('tempo_atendimento_seg') < 0) |
    (F.col('diff_abertura_ingestao_seg') < 0)
)

if df_ft_chamados_hora_tempo_dif.count() == 0:
    print('valores com tempo inconsistente: 0')
else:
    print(f'linhas com tempo inconsistente:')
    df_ft_chamados_hora_tempo_dif.limit(5).display()

In [0]:
df_ft_chamados_hora_irreg = df_ft_chamados_hora.filter(
    F.col('id_cliente').isNull() &
    F.col('id_chamado').isNull() &
    ~F.col('id_cliente').rlike(r'^[0-9]+$') &
    ~F.col('id_chamado').rlike(r'^[0-9]+$')
)

if df_ft_chamados_hora_irreg.count() == 0:
    print('linhas com valores irregulares: 0')
else:
    print(f'linhas com id_cliente, id_chamado irregulares:')
    df_ft_chamados_hora_irreg.limit(5).display()

### 3.4 Remoção de registros inconsistentes

- Removemos da tabela (`ft_chamados_hora`) todos os registros com valores temporais inconsistentes.

In [0]:
df_ft_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_tempo_dif)

print(f'linhas em chamados_hora apos remocao: {df_ft_chamados_hora.count()}')

### 3.5 Salvando dados tratados da tabela `ft_chamados_hora` na camada silver

- O DataFrame `df_ft_chamados_hora`, após todas as etapas de limpeza e transformação, é salvo na tabela `ft_chamados_hora` na camada silver utilizando o método `saveAsTable` com o modo `overwrite`, garantindo que os dados estejam atualizados.

In [0]:
save_table_silver('ft_chamados_hora', df_ft_chamados_hora)

df = read_table('ft_chamados_hora', 'silver')
describe_table(df)

## 4. Tratamento da tabela `ft_chamados`

### 4.1. Configuração e Leitura da Bronze
Defini as variáveis de ambiente `catalogo`, `bronze_db_name` e `silver_db_name` para organizar os caminhos do *Data Lake*.
Em seguida, realizei a leitura da tabela bruta (`df_bronze`). Como o arquivo original não possuía cabeçalho (gerando colunas genéricas como `_c0`), preparei o DataFrame para as transformações seguintes.

In [0]:
# Leitura da tabela Bronze de Chamados
df_bronze = spark.table(f"{catalogo}.{bronze_db_name}.ft_chamados")

describe_table(df_bronze)

### 4.2. Tratamento de Identificadores (IDs)
Nesta etapa, foquei na chave primária da tabela. Renomeei a coluna genérica `_c0` para `id_chamado`, seguindo as boas práticas de *snake_case*.
Para garantir a integridade dos dados, converti o campo para o tipo Inteiro (`int`) e apliquei a remoção de duplicatas (`dropDuplicates`), assegurando que cada chamado seja único na camada Silver.

In [0]:
df_ordenado = (
    df_bronze
    .withColumnRenamed("_c0", "id_chamado") #Renomeia a coluna
    .withColumn("id_chamado", col("id_chamado").cast("int")) #Transforma tudo em int
    .filter(col("id_chamado").isNotNull())  #Remove linhas se o ID estiver vazio (lixo)
    .dropDuplicates(["id_chamado"])         #Se tiver dois IDs iguais, mantém apenas um
    .orderBy("id_chamado")
)

df_ordenado.limit(5).display()

### 4.3. Normalização e Tipagem do ID Cliente
No dataframe `df_cliente_tratado`, renomeei a coluna para `id_cliente` (padrão *snake_case*).
Optei pela tipagem `long` para preservar a integridade de números grandes (como CPFs) e tratei os valores nulos preenchendo com `-1` (Cliente Desconhecido), garantindo que nenhum chamado fosse descartado por falta de identificação do cliente.

In [0]:
df_cliente_tratado = (
    df_ordenado # Continuando do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c1", "id_cliente")
    
    # 2. Tipagem SEGURA (Long em vez de Int para não quebrar CPFs)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # 3. Tratamento de Nulos (Regra de Ouro)
    # Não apaga a linha (o chamado existiu), mas marca o cliente como -1 (Desconhecido)
    .fillna(-1, subset=["id_cliente"])
)

df_cliente_tratado.limit(5).display()

### 4.4. Reconstrução da Coluna Motivo
No dataframe `df_motivo_tratado`, corrigi os erros de *encoding* (ex: "Contrata..o") utilizando Expressões Regulares (*Regex*).
Substituí os padrões corrompidos pelas palavras corretas e refinei a regra da palavra "Não" (usando `\b` para limites de palavra), evitando alterações indevidas em palavras como "Pontos". Finalizei padronizando o texto com a primeira letra maiúscula (*Initcap*).

In [0]:
df_motivo_tratado = (
    df_cliente_tratado # Continua do passo de ID_Cliente
    
    # 1. Renomear
    .withColumnRenamed("_c2", "motivo")
    
    # 2. Tipagem e Trim (Limpeza básica)
    .withColumn("motivo", trim(col("motivo").cast("string")))
    
    # 3. CIRURGIA DE RECONSTRUÇÃO (Regex)
    # O ponto (.) substitui o caractere estragado. 
    
    .withColumn("motivo", regexp_replace(col("motivo"), "Contrata..o", "Contratacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Contesta..o", "Contestacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Altera..o", "Alteracao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "cart.o", "cartao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "D.vidas", "Duvidas"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Informa..es", "Informacoes"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Solicita..o", "Solicitacao"))

    # \\b significa "borda da palavra". Só pega se começar e terminar ali.
    .withColumn("motivo", regexp_replace(col("motivo"), "(?i)\\bn.o\\b", "Nao"))
    
    # 4. Padronização Visual (Capitalize)
    # Deixa "duvidas gerais" -> "Duvidas Gerais"
    .withColumn("motivo", initcap(col("motivo")))
    
    # 5. Tratamento de Nulos
    # Regra: Motivo vazio vira "Motivo Nao Informado"
    .fillna("Motivo Nao Informado", subset=["motivo"])
    .withColumn("motivo", 
                when((col("motivo") == "") | (col("motivo").isNull()), "Motivo Nao Informado")
                .otherwise(col("motivo")))
)

df_motivo_tratado.limit(5).display()

### 4.5. Padronização de Canais
No dataframe `df_canal_tratado`, normalizei a escrita dos canais de atendimento (unificando "U.r.a" e "URA").
Implementei uma lógica hierárquica de regras (`when/otherwise`), priorizando a identificação de "Atendimento Especializado" antes de "Inicial" para evitar erros de classificação por *substrings*.

In [0]:
df_canal_tratado = (
    df_motivo_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c3", "canal")
    
    # 2. Tipagem e Trim
    .withColumn("canal", trim(col("canal").cast("string")))
    
    # 3. NORMALIZAÇÃO E PADRONIZAÇÃO
    .withColumn("canal", 
                
                # Regra 1: Chatbot
                when(lower(col("canal")).like("%chat%"), "Chatbot")
                
                # Regra 2: URA (Pega URA ou U.r.a)
                .when((lower(col("canal")).like("%ura%")) | (lower(col("canal")).like("%u.r.a%")), "URA")
                
                # Regra 3: Web e Email (Adicionei conforme sua lista)
                .when(lower(col("canal")).like("%web%"), "Web")
                .when(lower(col("canal")).like("%mail%"), "Email")
                
                # Regra 4: ATENDIMENTO ESPECIALIZADO (Checa ANTES do Inicial)
                # Se tiver a palavra "especializado" em qualquer lugar, classifica aqui
                .when(lower(col("canal")).like("%especializ%"), "Atendimento Especializado")
                
                # Regra 5: ATENDIMENTO INICIAL
                # Pega "Inicial", "Atend. Inicial", ou qualquer "Atend" genérico que sobrou
                .when((lower(col("canal")).like("%inici%")) | (lower(col("canal")).like("%atend%")), "Atendimento Inicial")
                
                .otherwise(initcap(col("canal")))
               )

    # 4. Tratamento de Nulos
    .fillna("Canal Nao Identificado", subset=["canal"])
    .withColumn("canal", 
                when((col("canal") == "") | (col("canal").isNull()), "Canal Nao Identificado")
                .otherwise(col("canal")))
)

# Validação Final
print("Validação: Verifique se Especializado e Inicial estão separados:")
df_canal_tratado.groupBy("canal").count().show(truncate=False)

df_canal_tratado.limit(5).display()

### 4.6. Flag Binária de Resolução
No dataframe `df_resolvido_tratado`, transformei a coluna de status em uma flag binária limpa.
Em vez de tratar acentos individualmente, usei uma lógica robusta que verifica a presença das letras "s" ou "n" (`like %s%`), blindando o código contra variações de escrita como "Sim", "SIM" ou erros de caracteres no "Não".

In [0]:
df_resolvido_tratado = (
    df_canal_tratado # Continua do passo anterior (Canal)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c4", "resolvido")
    
    # 2. Tipagem e Trim
    .withColumn("resolvido", trim(col("resolvido").cast("string")))
    
    # 3. NORMALIZAÇÃO BINÁRIA (melhor prática)
    # Estratégia: Em vez de brigar com o acento, usamos a lógica do "Contém S"
    .withColumn("resolvido", 
                
                # Regra 1: Se tiver "s" ou "S" (Sim, S, yes), vira "Sim"
                when(lower(col("resolvido")).like("%s%"), "Sim")
                
                # Regra 2: Se tiver "n" ou "N" (Nao, No, No), vira "Nao"
                .when(lower(col("resolvido")).like("%n%"), "Nao")
                
                # Caso contrário (Vazio ou Lixo), vira "Nao Informado"
                .otherwise("Nao Informado")
               )

    # 4. Tratamento de Nulos (Garantia Extra)
    # Se sobrar algum null real, vira "Nao Informado"
    .fillna("Nao Informado", subset=["resolvido"])
)

# Validação: Deve aparecer APENAS "Sim", "Nao" e talvez "Nao Informado"
print("Distribuição da coluna Resolvido:")
df_resolvido_tratado.groupBy("resolvido").count().show()

df_resolvido_tratado.limit(5).display()

### 4.7. Estruturação da Hora de Abertura
Nesta etapa, tratei exclusivamente a coluna `hora_abertura_chamado` (antiga `_c5`).
Embora os dados atuais estivessem vazios ou nulos, decidi forçar a tipagem imediata para `Timestamp` (*Schema Enforcement*). Essa decisão garante que a tabela Silver nasça com a estrutura correta de "Data e Hora" para receber dados futuros, evitando que a coluna permaneça como um texto genérico indefinido.

In [0]:
df_hora_abertura_tratado = (
    df_resolvido_tratado # Continua do passo anterior
    
    # 1. Renomear (Snake Case e Descritivo)
    .withColumnRenamed("_c5", "hora_abertura_chamado")
    
    # 2. Tipagem Forte (Schema Enforcement)
    # Mesmo que esteja tudo Null ou vazio, o tipo é Timestamp.
    .withColumn("hora_abertura_chamado", col("hora_abertura_chamado").cast("timestamp"))
)

# Validação (quero ver o schema como "timestamp" e os dados como "null")
print("Schema da coluna:")
df_hora_abertura_tratado.select("hora_abertura_chamado").printSchema()

print("\nVisualização dos dados (Devem estar null):")
df_hora_abertura_tratado.select("hora_abertura_chamado").show(5)

df_hora_abertura_tratado.limit(5).display()

### 4.8. Estruturação Temporal e Regra de Cópia
No dataframe `df_inicio_tratado`, forcei a tipagem das colunas de horário para `Timestamp`.
Implementei a regra de negócio para a **Hora de Início**: quando o registro indicava "igual a hora de abertura", o código copiou dinamicamente o valor da coluna anterior.

In [0]:
df_inicio_tratado = (
    df_hora_abertura_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c6", "hora_inicio_atendimento")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("hora_inicio_atendimento", trim(col("hora_inicio_atendimento")))
    
    # 3. Lógica de Negócio (A cópia condicional)
    # Se o texto disser "igual...", ele busca o valor da coluna hora_abertura_chamado.
    .withColumn("hora_inicio_atendimento", 
                when(lower(col("hora_inicio_atendimento")).like("%igual%"), col("hora_abertura_chamado"))
                .otherwise(col("hora_inicio_atendimento")))
    
    # 4. Tipagem Final (Schema Enforcement)
    # Tudo que não for data válida vira Null automaticamente aqui
    .withColumn("hora_inicio_atendimento", col("hora_inicio_atendimento").cast("timestamp"))
)

# Validação:
print("Schema atualizado:")
df_inicio_tratado.select("hora_inicio_atendimento").printSchema()

df_inicio_tratado.limit(5).display()

### 4.9. Estruturação da Hora de Finalização
Nesta etapa, tratei a coluna `hora_finalizacao_atendimento` (antiga `_c7`).
Realizei a limpeza de espaços em branco (*trim*) e forcei a conversão direta para o tipo `Timestamp`. Diferente da hora de início, não houve necessidade de regras condicionais complexas, mas a definição estrita do tipo garante que a tabela esteja tecnicamente preparada para receber os registros de tempo assim que estiverem disponíveis na origem.

In [0]:
df_fim_tratado = (
    df_inicio_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c7", "hora_finalizacao_atendimento")
    
    # 2. Trim (Limpeza básica de espaços invisíveis)
    .withColumn("hora_finalizacao_atendimento", trim(col("hora_finalizacao_atendimento")))
    
    # 3. Tipagem (Schema Enforcement)
    # Transforma texto/vazio em Data Real.
    .withColumn("hora_finalizacao_atendimento", col("hora_finalizacao_atendimento").cast("timestamp"))
)

# Validação do Schema
print("Schema final das colunas de tempo:")
df_fim_tratado.select("hora_abertura_chamado", 
                      "hora_inicio_atendimento", 
                      "hora_finalizacao_atendimento").printSchema()

df_fim_tratado.limit(5).display()

### 4.10. Sanitização do Tempo de Espera
No dataframe `df_espera_tratado`, tratei a métrica `tempo_espera_segundos`.
Identifiquei que a origem enviava a string "NULL", então apliquei uma sanitização para converter esse texto em nulo real antes da tipagem para `int`. Assumi valores nulos como `0` e corrigi eventuais números negativos para garantir a consistência dos cálculos de média futuros.

In [0]:
df_espera_tratado = (
    df_fim_tratado # Continua do passo anterior
    
    # 1. Renomear (coloquei _segundos pra especificar o tipo de tempo)
    .withColumnRenamed("_c8", "tempo_espera_segundos")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("tempo_espera_segundos", trim(col("tempo_espera_segundos")))
    
    # 3. SANITIZAÇÃO 
    # Antes de converter para número, removi a palavra "NULL" e vazios
    .withColumn("tempo_espera_segundos", 
                when((col("tempo_espera_segundos") == "NULL") | (col("tempo_espera_segundos") == ""), None)
                .otherwise(col("tempo_espera_segundos")))
    
    # 4. Tipagem
    .withColumn("tempo_espera_segundos", col("tempo_espera_segundos").cast("int"))
    
    # 5. Negativos viram 0
    .withColumn("tempo_espera_segundos", 
                when(col("tempo_espera_segundos") < 0, 0)
                .otherwise(col("tempo_espera_segundos")))
    
    # Nulos viram 0 para cálculo de média
    .fillna(0, subset=["tempo_espera_segundos"])
)

# Validação
print("Estatísticas do Tempo de Espera (Segundos):")
df_espera_tratado.select("tempo_espera_segundos").describe().show()

df_espera_tratado.limit(5).display()

### 4.11. Sanitização do Tempo de Conversa
No dataframe `df_conversa_tratado`, apliquei a mesma lógica de limpeza na coluna `tempo_conversa_segundos`.
Mantive os registros com duração "1" (mesmo sem *timestamps* válidos), preservando a informação de chamadas rápidas para análises de "Chamadas Fantasmas" na camada Gold.

In [0]:
df_conversa_tratado = (
    df_espera_tratado # Continua do passo anterior (Espera)
    
    # 1. Renomear
    .withColumnRenamed("_c9", "tempo_atendimento_segundos")
    
    # 2. Trim
    .withColumn("tempo_atendimento_segundos", trim(col("tempo_atendimento_segundos")))
    
    # 3. SANITIZAÇÃO (O Fix do "NULL" string)
    # Removemos a palavra escrita "NULL" antes de converter
    .withColumn("tempo_atendimento_segundos", 
                when((col("tempo_atendimento_segundos") == "NULL") | (col("tempo_atendimento_segundos") == ""), None)
                .otherwise(col("tempo_atendimento_segundos")))
    
    # 4. Tipagem (Integer)
    .withColumn("tempo_atendimento_segundos", col("tempo_atendimento_segundos").cast("int"))
    
    # 5. Regras de Sanidade
    # Negativos viram 0
    .withColumn("tempo_atendimento_segundos", 
                when(col("tempo_atendimento_segundos") < 0, 0)
                .otherwise(col("tempo_atendimento_segundos")))
    
    # Nulos viram 0 (Assumo zero conversa se estiver vazio)
    .fillna(0, subset=["tempo_atendimento_segundos"])
)

# Validação
print("Estatísticas do Tempo de Conversa (Segundos):")
df_conversa_tratado.select("tempo_atendimento_segundos").describe().show()

df_conversa_tratado.limit(5).display()

### 4.12. ID Atendente e Carga Final
No dataframe `df_final`, tratei a coluna `id_atendente`. Preenchi os valores nulos com `-1`, criando a categoria "Atendimento Automático" para evitar perdas em cruzamentos com a tabela de funcionários.
Por fim, gravei o resultado na tabela `chamados` do banco `silver_db_name`, utilizando o formato Delta com sobrescrita (`mode("overwrite")`) para atualizar a camada Silver.

In [0]:
df_final = (
    df_conversa_tratado # Continua do passo anterior (Tempo Conversa)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c10", "id_atendente")
    
    # 2. Trim
    .withColumn("id_atendente", trim(col("id_atendente")))
    
    # 3. SANITIZAÇÃO (Limpa a string "NULL" e vazios)
    .withColumn("id_atendente", 
                when((col("id_atendente") == "NULL") | (col("id_atendente") == ""), None)
                .otherwise(col("id_atendente")))
    
    # 4. Tipagem (Integer)
    .withColumn("id_atendente", col("id_atendente").cast("int"))
    
    # 5. Tratamento de Nulos
    # Se estiver vazio, coloquei -1 (Indica URA/Bot ou erro de sistema)
    .fillna(-1, subset=["id_atendente"])
)

# Validação
print("Amostra dos IDs de Atendente:")
df_final.select("id_atendente").distinct().show(10)

save_table_silver("ft_chamados", df_final)

describe_table(df_final)

## 5. Tratamento da tabela `ft_custos`

### Descrição
Este notebook realiza a transformação de dados da camada **Bronze** para **Silver** da tabela `custos` (origem) para `vcredit_custos` (destino).

### Adaptação de Schema
Como a ingestão Bronze foi realizada sem cabeçalho, as colunas originais (`_c0`, `_c1`, `_c2`) serão renomeadas para nomes de negócio (`id_custo`, `id_chamado`, `custo`) logo no início do processamento.

### Objetivos
* Renomear colunas técnicas para nomes de negócio
* Padronizar a coluna `custo` (remover texto "reais" e corrigir separadores)
* Converter tipo de dados de String para Decimal
* Garantir unicidade pela chave primária
* Salvar na camada Silver em formato Delta Lake

In [0]:
tabela_origem = "ft_custos" 

# 1. Carregar a tabela bruta
df_bronze_raw = read_table(tabela_origem, "bronze")

# 2. Renomear colunas genéricas para nomes de negócio
# _c0 -> id_custo
# _c1 -> id_chamado
# _c2 -> custo
df_bronze = df_bronze_raw \
    .withColumnRenamed("_c0", "id_custo") \
    .withColumnRenamed("_c1", "id_chamado") \
    .withColumnRenamed("_c2", "custo")

display(df_bronze.limit(5))

### Análise Exploratória
Agora com as colunas renomeadas, analisamos o schema e uma amostra dos dados para confirmar os padrões de sujeira na coluna de valor.

In [0]:
# Schema e amostra
describe_table(df_bronze)

# Verificando padrões na coluna 'custo' (ex: '0.0026reais' vs '0,1816')
print("Amostra da coluna 'custo' (dados brutos):")
display(df_bronze.select("custo").sample(withReplacement=False, fraction=0.1).limit(10))

### Problemas Identificados
1.  **Formato Inconsistente:** A coluna `custo` mistura formatos. Alguns registros possuem o sufixo "reais" e ponto, outros usam vírgula como decimal.
2.  **Tipo Incorreto:** Dados numéricos estão tipados como `string`.
3.  **Nomenclatura:** O nome `custo` é genérico, vamos alterar para `valor_custo`.

---
### Transformações e Limpeza
Aplicamos a limpeza agressiva (Regex) para remover textos e padronizar o formato numérico, além de ajustar a tipagem.

In [0]:
# Tratamento da coluna de valor
df_step1 = df_bronze.select(
    F.col("id_custo"),
    F.col("id_chamado"),
    
    # Remove tudo que não for número/ponto/vírgula, troca ',' por '.' e converte
    F.regexp_replace(
        F.regexp_replace(F.col("custo"), "[^0-9,.]", ""), 
        ",", "."
    ).cast(DecimalType(18, 6)).alias("valor_custo"),
    
    # Mantendo a coluna de controle da equipe
    F.col("data_ingestao").alias("ingestion_timestamp")
)

print("Amostra após limpeza:")
describe_table(df_step1)

### Tratamento de Duplicatas e Ordenação
Garantia de integridade da chave primária.

In [0]:
# --- Passo 2: Tratamento de Duplicatas ---
# Contagem antes
total_antes = df_step1.count()

# Remover duplicatas pelo ID (Chave Primária)
# Criamos o df_silver final a partir daqui
df_silver = df_step1.dropDuplicates(["id_custo"])

# Contagem depois
total_depois = df_silver.count()

print(f"Registros antes: {total_antes}")
print(f"Registros depois: {total_depois}")
print(f"Duplicatas removidas: {total_antes - total_depois}")

# --- Passo 3: Ordenação ---
df_silver = df_silver.orderBy(F.col("id_custo"))

describe_table(df_silver)

### Análise Completa de Qualidade - Custos
Agora que os dados estão limpos, calculamos métricas de qualidade e estatísticas financeiras para garantir que não perdemos dados importantes e que os valores fazem sentido.

In [0]:
# Análise completa de qualidade e estatísticas financeiras
qualidade_dados = df_silver.select([
    F.count("*").alias("total_registros"),
    F.countDistinct("id_custo").alias("id_custo_unicos"),
    F.countDistinct("id_chamado").alias("id_chamado_unicos"),
    F.avg("valor_custo").alias("custo_medio"),
    F.sum("valor_custo").alias("custo_total"),
    F.min("valor_custo").alias("custo_minimo"),
    F.max("valor_custo").alias("custo_maximo")
]).collect()[0]

print("RELATÓRIO DE QUALIDADE:")
print(f"Total de registros: {qualidade_dados['total_registros']}")
print(f"IDs custo únicos: {qualidade_dados['id_custo_unicos']}")
print(f"IDs chamado únicos: {qualidade_dados['id_chamado_unicos']}")
print("-" * 30)
print("ESTATÍSTICAS FINANCEIRAS:")
print(f"Custo Médio: R$ {qualidade_dados['custo_medio']:.2f}")
print(f"Custo Mínimo: R$ {qualidade_dados['custo_minimo']:.2f}")
print(f"Custo Máximo: R$ {qualidade_dados['custo_maximo']:.2f}")
print(f"Investimento Total Monitorado: R$ {qualidade_dados['custo_total']:.2f}")

# Verificar integridade 
nulos = df_silver.filter(F.col("id_chamado").isNull()).count()
print("-" * 30)
print(f"Registros órfãos (sem id_chamado): {nulos}")

In [0]:
save_table_silver("ft_custos", df_silver)

In [0]:
describe_table(df_silver)

## 6. Tratamento da tabela `ft_clientes`

### 6.1. Configuração e Leitura da Bronze
Defini as variáveis de ambiente `catalogo`, `bronze_db_name` e `silver_db_name` para organizar os caminhos do *Data Lake*.
Em seguida, realizei a leitura da tabela bruta (`df_bronze`). Como o arquivo original não possuía cabeçalho (gerando colunas genéricas como `_c0`), preparei o DataFrame para as transformações seguintes.

In [0]:
# Leitura da tabela Bronze de Clientes
df_bronze = read_table("ft_clientes")

describe_table(df_bronze)

### 6.2. Tratamento do ID Cliente (CPF)
Tratei a coluna `_c0`, renomeando-a para `id_cliente`.
Optei pela tipagem `long` para suportar os 11 dígitos do CPF sem truncagem. Apliquei a remoção de duplicados (`dropDuplicates`) e filtrei valores nulos, garantindo a unicidade da chave primária.

In [0]:
df_step_id = (
    df_bronze
    .withColumnRenamed("_c0", "id_cliente")
    
    # Tipagem Long (CPFs são números grandes)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # Regras de Integridade
    .dropDuplicates(["id_cliente"])
    .filter(col("id_cliente").isNotNull())
)

describe_table(df_step_id)

%md
### 6.3. Tratamento do Nome
Tratei a coluna `_c1`, renomeando-a para `nome_cliente`.
Apliquei as funções `trim` para remover espaços extra e `initcap` para padronizar o texto em *Title Case* (ex: "João Silva"), corrigindo possíveis inconsistências de maiúsculas/minúsculas vindas da origem.

In [0]:
df_step_nome = (
    df_step_id # Continua do passo anterior
    .withColumnRenamed("_c1", "nome_cliente")
    
    # Padronização de Texto
    .withColumn("nome_cliente", trim(col("nome_cliente")))
    .withColumn("nome_cliente", initcap(col("nome_cliente")))
)

describe_table(df_step_nome)

### 6.4. Tratamento e Validação do E-mail
Tratei a coluna `_c2`, renomeando-a para `email_cliente`.
Padronizei todos os caracteres para minúsculo (`lower`). Implementei uma regra de qualidade simples: se o e-mail não contiver o caractere "@", o valor é convertido para `null` (considerado inválido), evitando "lixo" nos dados de contacto.

In [0]:
df_step_email = (
    df_step_nome # Continua do passo anterior
    .withColumnRenamed("_c2", "email_cliente")
    
    # Padronização (Minúsculo)
    .withColumn("email_cliente", trim(lower(col("email_cliente"))))
    
    # Validação de Formato
    .withColumn("email_cliente", 
                when(col("email_cliente").contains("@"), col("email_cliente"))
                .otherwise(None))
)

describe_table(df_step_email)

### 6.5 Tratamento da Região
Tratei a coluna `_c3`, renomeando-a para `regiao`.
Apliquei a limpeza de espaços e a padronização visual (*Title Case*) para garantir a consistência nos filtros e agrupamentos geográficos na camada Gold.

In [0]:
df_step_regiao = (
    df_step_email # Continua do passo anterior
    .withColumnRenamed("_c3", "regiao")
    
    # Padronização
    .withColumn("regiao", initcap(trim(col("regiao"))))
)

display(
    df_step_regiao
    .select('regiao')
    .distinct()
)

describe_table(df_step_regiao)

### 6.6. Tratamento e Sanidade da Idade
Tratei a coluna `_c4`, renomeando-a para `idade`.
Forcei a conversão para o tipo Inteiro (`int`) e apliquei uma regra de sanidade (*sanity check*): valores negativos ou absurdamente altos (acima de 120 anos) foram convertidos para nulo, garantindo métricas demográficas fiáveis.

In [0]:
df_step_idade = (
    df_step_regiao # Continua do passo anterior
    .withColumnRenamed("_c4", "idade")
    
    # Tipagem
    .withColumn("idade", col("idade").cast("int"))
    
    # Regra de Sanidade (0 < Idade < 120)
    .withColumn("idade", 
                when((col("idade") > 0) & (col("idade") < 120), col("idade"))
                .otherwise(None))
)

describe_table(df_step_idade)

### 6.7. Auditoria e Gravação Silver
Finalizei o processo adicionando os metadados de controle. Preservei a `data_ingestao` original (renomeada para `_bronze`) e criei a `data_ingestao_silver` com o *timestamp* atual.
Gravei a tabela processada no catálogo `medalhao_credit`, esquema `silver`, em formato Delta, garantindo a persistência e qualidade dos dados de clientes.

In [0]:
# Preparação Final
df_final = df_step_idade

# Gravação
save_table_silver("ft_clientes", df_final)

## 7. Tratamento da tabela `ft_pesquisa`

### Descrição
Este notebook realiza a transformação de dados da camada **Bronze** para **Silver** da tabela `pesquisa_satisfacao`, incluindo limpeza, engenharia de features e garantia de qualidade.

### Objetivos
- Renomear colunas para nomes descritivos
- Converter tipo de dados da coluna `nota_atendimento`
- Tratar valores nulos
- Criar categoria de nota (Insatisfeito, Neutro, Satisfeito, Não respondeu)
- Salvar na camada Silver em formato Delta Lake

### 7.1 Importando bibliotecas e carregando tabela

In [0]:
path_pesquisa = f"{catalogo}.{bronze_db_name}.ft_pesquisa_satisfacao"

df_bronze = read_table("ft_pesquisa_satisfacao", "bronze")
describe_table(df_bronze)

### 7.2 Análise Exploratória
Neste trecho iremos analisar o schema, uma amostra da tabela bem como suas estatísticas com a função describe, por fim, iremos ver a distribuição das notas.


In [0]:
# Schema e amostra
describe_table(df_bronze)

# Estatísticas das notas
display(df_bronze.describe())

# Distribuição das notas
display(df_bronze.groupBy("_c2").count().orderBy("_c2"))

#### Problemas Identificados
1. **Nomes de colunas não descritivos**
2. **Tipo incorreto**: `nota_atendimento` como string
3. **Valores nulos**: 2.700 registros

### 7.3. Transformações Aplicadas

#### 7.3.1. Renomeação de Colunas

In [0]:
df_bronze = (df_bronze
    .withColumnRenamed("_c0", "id_pesquisa")
    .withColumnRenamed("_c1", "id_chamado") 
    .withColumnRenamed("_c2", "nota_atendimento")
)

display(df_bronze.limit(10))

#### 7.3.2. Conversão de Tipo de Dados

In [0]:
df_bronze = df_bronze.withColumn(
    "nota_atendimento",
    F.when(F.col("nota_atendimento") == "NULL", F.lit(None)) 
    .otherwise(F.col("nota_atendimento").cast(IntegerType()))
)

In [0]:
df_bronze.printSchema()

#### 7.3.3. Ordenação
Ordenado por id_pesquisa para melhor organização

In [0]:
# Ordenar por id_pesquisa
df_bronze = df_bronze.orderBy(F.col("id_pesquisa"))

display(df_bronze.limit(20))

### 7.4 Análise Completa de Qualidade - Pesquisa de Satisfação

In [0]:
df_respondentes = df_bronze.filter(F.col("nota_atendimento").isNotNull())

qualidade_respondentes = df_respondentes.select([
    F.count("*").alias("total_respondentes"),
    F.countDistinct("id_chamado").alias("id_chamado_unicos"),
    F.countDistinct("id_pesquisa").alias("id_pesquisa_unicos"),
    F.avg("nota_atendimento").alias("nota_media_respondentes"),
    F.min("nota_atendimento").alias("nota_minima_respondentes"),
    F.max("nota_atendimento").alias("nota_maxima_respondentes"),
    F.expr("percentile_approx(nota_atendimento, 0.5)").alias("mediana_respondentes"),
    F.stddev("nota_atendimento").alias("desvio_padrao_respondentes")
]).collect()[0]

total_geral = df_bronze.count()
total_nao_respondentes = df_bronze.filter(F.col("nota_atendimento").isNull()).count()

taxa_resposta = (qualidade_respondentes['total_respondentes'] / total_geral) * 100

print("Relatório de qualidade - quem respondeu:")
print(f"Estatísticas")
print(f"Total de registros: {total_geral}")
print(f"Responderam: {qualidade_respondentes['total_respondentes']} ({taxa_resposta:.1f}%)")
print(f"Não responderam: {total_nao_respondentes} ({(total_nao_respondentes/total_geral)*100:.1f}%)")

print("\nESTATÍSTICAS DOS RESPONDENTES:")
print(f"IDs chamado únicos: {qualidade_respondentes['id_chamado_unicos']}")
print(f"IDs pesquisa únicos: {qualidade_respondentes['id_pesquisa_unicos']}")
print(f"Nota média: {qualidade_respondentes['nota_media_respondentes']:.2f}")
print(f"Mediana: {qualidade_respondentes['mediana_respondentes']:.2f}")
print(f"Desvio padrão: {qualidade_respondentes['desvio_padrao_respondentes']:.2f}")
print(f"Range notas: [{qualidade_respondentes['nota_minima_respondentes']}, {qualidade_respondentes['nota_maxima_respondentes']}]")

### 7.5. Categorização das Notas
- **(1 a 2)**: Insatisfeito
- **3**: Neutro
- **(4 a 5)**: Satisfeito
- **null**: Não respondeu

In [0]:
df_bronze = df_bronze.withColumn(
    "categoria_nota",
    F.when(F.col("nota_atendimento").between(1, 2), "Insatisfeito")
     .when(F.col("nota_atendimento") == 3, "Neutro")
     .when(F.col("nota_atendimento").between(4, 5), "Satisfeito")
     .when(F.col("nota_atendimento").isNull(), "Não Respondeu")
     .otherwise("Indefinido")
)

print("Distribuição por categoria:")
display(df_bronze.groupBy("categoria_nota").count().orderBy("count", ascending=False))

In [0]:
describe_table(df_bronze)

### 7.6. Conclusão
- **Dados íntegros**: Sem duplicatas ou gaps
- **Qualidade alta**: Todos os problemas tratados
- **Pronto para análise**: Categorias criadas e dados limpos

#### 7.6.1 Salvando na silver_credit

In [0]:
# Salvando na silver
save_table_silver("ft_pesquisa_satisfacao", df_bronze)

## 8. Tratamento das tabelas `dm_base_atendentes`, `dm_base_motivos` e` dm_canais`

### 8.1 Importando tabelas

In [0]:
bronze_atendentes = f"{bronze_db_name}.dm_base_atendentes"
bronze_canais = f"{bronze_db_name}.dm_canais"
bronze_motivos = f"{bronze_db_name}.dm_base_motivos"
silver_atendentes = f"{silver_db_name}.dm_base_atendentes"
silver_canais = f"{silver_db_name}.dm_canais"
silver_motivos = f"{silver_db_name}.dm_base_motivos"

In [0]:
df_atendentes = read_table("dm_base_atendentes", "bronze")
df_canais = read_table("dm_canais", "bronze")
df_motivos = read_table("dm_base_motivos", "bronze")

### 8.2. Tabela Atendentes
* Checagem feitas:
    * Total de Valores Nulos: 0
    * Total de Valores Duplicados: 0
    * Os `id_atendente `vão de 1 a 20
    * A coluna `nivel_atendimento` tem apenas valores 1 e 2
    * Mudança dos nomes das colunas

In [0]:
df_atendentes = df_atendentes \
    .withColumnRenamed("_c0", "id_atendente") \
    .withColumnRenamed("_c1", "nome_atendente") \
    .withColumnRenamed("_c2", "nivel_atendimento")

In [0]:
describe_table(df_atendentes)

In [0]:
nulos_df_atendentes = df_atendentes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_atendentes.columns
]).display()

In [0]:
len_atendentes = df_atendentes.count()
df_atendentes = df_atendentes.dropDuplicates(["id_atendente"])
len_after = df_atendentes.count()

print(f"Total Duplicatas: {len_atendentes - len_after}")

In [0]:
df_gabarito = spark.range(1, 21).toDF("id")
df_erros_status = df_atendentes.filter(~F.col("nivel_atendimento").isin([1, 2]))
df_check = df_atendentes.select("id_atendente").distinct()
ids_faltantes = df_gabarito.subtract(df_check)

if ids_faltantes.count() == 0:
    print("Sucesso: Todos os IDs de 1 a 20 estão presentes.")
else:
    print("Atenção: A sequência está furada. Faltam os seguintes IDs:")
    ids_faltantes.orderBy("id").display()

if df_erros_status.count() > 0:
    print(f"Atenção! Encontramos {df_erros_status.count()} linhas com status inválido:")
    df_erros_status.display() 
else:
    print("Sucesso: A coluna status contém apenas valores 1 e 2.")

In [0]:
# df_atendentes = df_atendentes \
#                 .withColumnRenamed("data_ingestao","data_ingestao_bronze")

In [0]:
save_table_silver("dm_base_atendentes", df_atendentes)

### 8.3 Tabela Canais
* Checagem feitas:
    * Total de Valores Nulos: 0
    * Total de Valores Duplicados: 0
    * Mudança dos nomes das colunas
    * Padronização da coluna `status_canal`: Inativo estava escrito errado em alguns casos.
    * Padronização da coluna `nome_canal` : Email não estava em maiúsculo.

In [0]:
df_canais = df_canais \
    .withColumnRenamed("_c0", "nome_canal") \
    .withColumnRenamed("_c1", "status_canal") 

In [0]:
describe_table(df_canais)

In [0]:
df_canais_tratado = df_canais \
   .withColumn("nome_canal", 
        F.when(F.lower(F.col("nome_canal")).like("%ura%"), "URA") \
        .otherwise(F.initcap(F.col("nome_canal")))
    ) \
    .withColumn("status_canal", F.regexp_replace(F.col("status_canal"), "invativo", "inativo")) \
    .withColumn("data_ingestao_silver", F.current_timestamp()) \
    .withColumnRenamed("data_ingestao", "data_ingestao_bronze") \
    
df_canais_tratado.limit(5).display()

In [0]:
save_table_silver("dm_canais", df_canais_tratado)

Como já podemos observar manualmente, não há valores nulos ou valores duplicados.

### 8.4 Tabela Motivos
* Checagens feitas:
    * Total de Valores Nulos: 0
    * Total de Valores Duplicados: 0
    * Mudança dos nomes das colunas
    * Mudar `id_motivo` de 3 a 15 para 1 a 13 para garantir consistência
    * Preencher coluna `categoria` que veio nula com categorias Financeiro, Cartão, Cadastral e Outros
    * Deixar coluna `criticidade` padronizada (alguns valores vieram com inicial maiúscula e outros minúscula)
    * Corrigir "Compra no autorizada" para "Compra não autorizada".

In [0]:
df_motivos = df_motivos \
    .withColumnRenamed("_c0", "id_motivo") \
    .withColumnRenamed("_c1", "nome_motivo") \
    .withColumnRenamed("_c2", "categoria") \
    .withColumnRenamed("_c3", "criticidade")

In [0]:
describe_table(df_motivos)

Como já podemos observar manualmente, não há valores nulos ou valores duplicados. Porém percebemos **uma coluna inteiramente nula** (que era para estar preenchida com **Financeiro, Cartão ou Cadastral**),  **inconsistências nas categorias ["baixa", "Baixa"] e ID's inconsistentes (de 3 a 15).**

In [0]:
janela_ordenacao = Window.orderBy("id_motivo")

df_motivos_tratado = df_motivos \
    .withColumn("criticidade", F.regexp_replace(F.initcap(F.col("criticidade")), "Média", "Media")) \
    .withColumn("id_motivo", F.row_number().over(janela_ordenacao)) \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Compra no autorizada", "Compra Nao Autorizada"))\
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Contrata..o", "Contratacao")) \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Contesta..o", "Contestacao")) \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Altera..o", "Alteracao"))   \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "cart.o", "Cartao"))   \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "D.vidas", "Duvidas"))  \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Informa..es", "Informacoes"))  \
    .withColumn("nome_motivo", F.regexp_replace(F.col("nome_motivo"), "Solicita..o", "Solicitacao"))   \
    .withColumn("nome_motivo", F.initcap(F.col("nome_motivo"))) \
    .withColumn("categoria", 
        F.when(F.lower(F.col("nome_motivo")).like("%cadastrais%"), "Cadastral")
        .when(F.lower(F.col("nome_motivo")).like("%dados%"), "Cadastral")
        .when(F.lower(F.col("nome_motivo")).like("%fatura%"), "Financeiro")
        .when(F.lower(F.col("nome_motivo")).like("%dívida%"), "Financeiro")
        .when(F.lower(F.col("nome_motivo")).like("%agência%"), "Cadastral")
        .when(F.lower(F.col("nome_motivo")).like("%limite%"), "Cartao")
        .when(F.lower(F.col("nome_motivo")).like("%cartão%"), "Cartao")
        .when(F.lower(F.col("nome_motivo")).like("%compra%"), "Cartao")
        .otherwise("Outros") 
    ) \
    .withColumn("data_ingestao_silver", F.current_timestamp()) \
    .withColumnRenamed("data_ingestao", "data_ingestao_bronze") 
describe_table(df_motivos_tratado)

In [0]:
save_table_silver("dm_base_motivos", df_motivos_tratado)

Nesta etapa, carregamos todas as tabelas necessárias para a construção da nossa visão consolidada. Inicialmente, listamos as entidades envolvidas, identificando (nos comentários) as chaves primárias ou estrangeiras que seriam fundamentais para os cruzamentos.

Em seguida, instanciamos cada tabela da camada `silver_credit` em seu próprio DataFrame do Spark. Trouxemos para a memória tanto as tabelas dimensionais (como `base_atendentes`, `clientes`, `canais` e `base_motivos`) quanto as tabelas fato (`chamados`, `custos`, `pesquisa_satisfacao`), preparando o ambiente para a execução dos *joins* e o enriquecimento dos dados.

## 9. Consolidação da Visão Geral (Criação da Tabela `chamados_geral`)

Nesta etapa central do pipeline, realizamos a construção do DataFrame `df_chamados_geral`, que serve como a nossa "Tabela Unificada" (*One Big Table*) para análises.

**Leitura das Tabelas na Camada Silver:**
Optamos por realizar a leitura das tabelas nesta etapa para garantir um *sanity check* dos dados e também para permitir que a criação da tabela `chamados_geral` possa ser executada isoladamente, sem a necessidade de reprocessar todas as etapas anteriores do pipeline (por razões de testagem).

**Estratégia de Joins:**
Utilizamos a tabela `df_chamados` como ponto de partida, aplicando um `right join` com `df_chamados_hora` para garantir que a granularidade temporal fosse preservada como a espinha dorsal do *dataset*. Em seguida, enriquecemos esses dados através de múltiplos `left joins` com as tabelas dimensionais (`atendentes`, `motivos`, `canais`, `clientes`) e tabelas satélites (`custos`, `pesquisa_satisfacao`), consolidando métricas e descrições num único local.

**Seleção e Tratamento de Colunas:**
Durante a projeção dos campos finais (`select`):
* Resolvemos ambiguidades de colunas presentes em múltiplas tabelas (como `id_cliente`), especificando explicitamente a origem através de *aliases*.
* Mapeamos colunas de negócio, observando (conforme comentado no código) que campos como `categoria` e `criticidade` retornaram valores nulos nesta carga, mas foram mantidos para preservar o esquema.
* Adicionamos a coluna `data_criacao_silver` com o *timestamp* atual para rastreabilidade da geração desta tabela consolidada.
* Ordenamos o resultado por `id_chamado` para facilitar a leitura sequencial e exibimos o DataFrame consolidado.
* Como etapa final deste *pipeline*, materializamos o DataFrame `df_chamados_geral` no armazenamento físico do Data Lake.


In [0]:
df_base_atendentes = read_table("dm_base_atendentes", "silver")
df_base_motivos = read_table("dm_base_motivos", "silver")
df_canais = read_table("dm_canais", "silver")
df_chamados = read_table("ft_chamados", "silver")
df_chamados_hora = read_table("ft_chamados_hora", "silver")
df_clientes = read_table("ft_clientes", "silver")
df_custos = read_table("ft_custos", "silver")
df_pesquisa_satisfacao = read_table("ft_pesquisa_satisfacao", "silver")

In [0]:
df_chamados_geral = (
    df_chamados
    .join(df_chamados_hora, "id_chamado", "right")
    .join(df_base_atendentes, "id_atendente", "left")
    .join(df_base_motivos, df_chamados["motivo"] == df_base_motivos["nome_motivo"], "left")
    .join(df_canais, df_chamados["canal"] == df_canais["nome_canal"], "left")
    .join(df_clientes, "id_cliente", "left")
    .join(df_custos, "id_chamado", "left")
    .join(df_pesquisa_satisfacao, "id_chamado", "left")
    .select(
        "id_chamado",
        df_chamados["id_cliente"].alias("id_cliente"),
        "motivo",
        "categoria",
        "categoria_nota",
        "nota_atendimento",
        "criticidade",
        "canal",
        "status_canal",
        "resolvido",
        df_chamados_hora["hora_abertura_chamado"],
        df_chamados_hora["hora_inicio_atendimento"],
        df_chamados_hora["hora_finalizacao_atendimento"],
        "tempo_espera_segundos",
        "tempo_atendimento_segundos",
        "id_atendente",
        "nome_atendente",
        "nivel_atendimento",
        "valor_custo",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade",
        current_timestamp().alias("data_criacao_silver")
    ).orderBy('id_chamado')
)

describe_table(df_chamados_geral)

In [0]:
save_table_silver("ft_chamados_geral", df_chamados_geral, process_col=False)